# 高级灰度变换与直方图规定化

**实验目的：**
1. 掌握比特平面分层（Bit-plane Slicing）技术，理解不同位平面的信息含量差异。
2. 学会手动计算累积分布函数（CDF），不依赖现成匹配接口实现直方图规定化（匹配）。
3. 完成两幅图像的影调迁移，观察“源图风格”向“目标图风格”的灰度分布靠拢过程。

**实验内容：**
1. 将 8-bit 灰度图拆分为 8 个二进制位平面，对比高低阶平面的信息含量。
2. 不使用直接接口，手动计算 CDF，实现直方图规定化（匹配），完成两幅图像的影调迁移。

## 一、基础知识介绍

### 1）比特平面分层（Bit-plane Slicing）
- 8-bit 灰度图像每个像素由 8 个二进制位组成（bit7 到 bit0）。
- 高位平面（如 bit7、bit6）通常保留主要轮廓与亮暗结构。
- 低位平面（如 bit1、bit0）通常更接近纹理细节或噪声信息。

### 2）直方图与累积分布函数（CDF）
- 灰度直方图描述每个灰度值的出现频率。
- 将直方图归一化得到概率分布函数（PDF）。
- 对 PDF 累加可得累积分布函数（CDF）。

### 3）直方图规定化（Histogram Specification / Matching）
- 目标：让源图像的灰度分布尽量接近目标图像分布。
- 思路：比较源图 CDF 与目标图 CDF，建立灰度映射关系。
- 本实验要求手动实现，不调用 `match_histograms` 这类直接接口。

## 二、应用场景

1. **医学影像增强**：通过灰度分布迁移提升病灶与背景对比度。
2. **遥感影像标准化**：把不同时相、不同设备采集图像调整到近似灰度风格。
3. **安防与工业检测**：先做影调匹配可减小光照差异对后续识别算法的影响。
4. **文档与老照片修复**：通过灰度映射改善视觉层次、提升可读性。

## 三、实验部分

本实验按“准备数据 → 比特平面分层 → 手写 CDF 规定化 → 结果分析”的顺序进行。

- 所有代码单元前均给出说明。
- 建议按单元顺序执行。
- 若网络不可用，代码会自动回退到内置图像样本，保证实验可运行。

### 3.1 环境准备
导入实验依赖并设置绘图参数与中文字体回退，避免中文标题乱码。

In [3]:
import os
import urllib.request
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

# 固定随机种子，保证实验可复现
np.random.seed(42)

# 绘图参数设置
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['image.cmap'] = 'gray'
plt.rcParams['font.sans-serif'] = [
    'Microsoft YaHei', 'SimHei', 'SimSun',
    'Noto Sans CJK SC', 'Arial Unicode MS', 'DejaVu Sans'
]
plt.rcParams['axes.unicode_minus'] = False

### 3.2 下载并准备两张真实图像
本单元尝试从网上下载两张真实灰度图：一张作为源图，一张作为目标图；若网络不可用，会自动回退到 `skimage` 内置图像。

In [4]:
# 用于保存下载图像的目录
data_dir = 'data'
os.makedirs(data_dir, exist_ok=True)

# 两组在线真实图像 URL（优先使用 picsum，失败后用 OpenCV 样例图兜底）
source_url_candidates = [
    'https://picsum.photos/seed/dip-source-2026/640/640',
    'https://raw.githubusercontent.com/opencv/opencv/master/samples/data/lena.jpg'
]
target_url_candidates = [
    'https://picsum.photos/seed/dip-target-2026/640/640',
    'https://raw.githubusercontent.com/opencv/opencv/master/samples/data/sudoku.png'
]

source_path = os.path.join(data_dir, 'source_online_image.png')
target_path = os.path.join(data_dir, 'target_online_image.png')


def download_first_available(url_candidates, file_path):
    """按顺序尝试多个 URL，任意一个成功即返回 (True, 成功URL)。"""
    for url in url_candidates:
        try:
            urllib.request.urlretrieve(url, file_path)
            return True, url
        except Exception as error:
            print(f'下载失败: {url}')
            print(f'原因: {error}')
    return False, None


def load_gray_uint8(path):
    """加载图像并转为 8-bit 灰度 ndarray。"""
    image = Image.open(path).convert('L')
    return np.array(image, dtype=np.uint8)


# 尝试联网下载
ok_source, used_source_url = download_first_available(source_url_candidates, source_path)
ok_target, used_target_url = download_first_available(target_url_candidates, target_path)

if ok_source and ok_target:
    source_image = load_gray_uint8(source_path)
    target_image = load_gray_uint8(target_path)
    image_source_note = 'online images'
    print('源图URL:', used_source_url)
    print('目标图URL:', used_target_url)
else:
    # 网络不可用时回退到内置样本
    try:
        from skimage import data
        source_image = data.camera().astype(np.uint8)
        target_image = data.moon().astype(np.uint8)
        image_source_note = 'skimage built-in fallback'
    except Exception:
        # 兜底：若 skimage 也不可用，则用简单仿真图
        source_image = np.tile(np.linspace(0, 255, 512, dtype=np.uint8), (512, 1))
        target_image = np.flipud(source_image)
        image_source_note = 'synthetic fallback'

print('图像来源:', image_source_note)
print('源图尺寸:', source_image.shape, '目标图尺寸:', target_image.shape)

源图URL: https://picsum.photos/seed/dip-source-2026/640/640
目标图URL: https://picsum.photos/seed/dip-target-2026/640/640
图像来源: online images
源图尺寸: (640, 640) 目标图尺寸: (640, 640)


### 3.3 展示源图与目标图（说明）
先观察两张图像的视觉风格差异（亮度、对比度、暗部比例），为后续“影调迁移”建立直观认识。

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].imshow(source_image, vmin=0, vmax=255)
axes[0].set_title('源图（待匹配）')
axes[0].axis('off')

axes[1].imshow(target_image, vmin=0, vmax=255)
axes[1].set_title('目标图（提供影调）')
axes[1].axis('off')

plt.tight_layout()
plt.show()

### 3.4 实验1：比特平面分层函数与信息量指标（说明）
本单元实现：
1. 8 个位平面提取；
2. 位平面重建（只保留高位或低位）；
3. 每个位平面的 1 比例与二值熵（用于衡量信息活跃程度）。

In [ ]:
def bit_plane_slicing(image_uint8):
    """将 8-bit 灰度图拆分为 8 个位平面（bit0~bit7）。"""
    planes = []
    for bit in range(8):
        plane = ((image_uint8 >> bit) & 1).astype(np.uint8)
        planes.append(plane)
    return planes


def reconstruct_from_bits(planes, bit_indices):
    """根据指定位平面索引重建图像。"""
    reconstructed = np.zeros_like(planes[0], dtype=np.uint16)
    for bit in bit_indices:
        reconstructed += (planes[bit].astype(np.uint16) << bit)
    return np.clip(reconstructed, 0, 255).astype(np.uint8)


def binary_entropy(binary_plane):
    """计算二值图熵：H = -p log2 p - (1-p) log2 (1-p)。"""
    p = float(np.mean(binary_plane))
    if p <= 0.0 or p >= 1.0:
        return 0.0
    return -(p * np.log2(p) + (1 - p) * np.log2(1 - p))

### 3.5 实验1执行：可视化 8 个位平面并比较高低位贡献（说明）
重点观察：
- 高位（bit7~bit4）是否更接近原图结构；
- 低位（bit3~bit0）是否更多表现细碎纹理或噪声；
- 重建图中“只保留高位”与“只保留低位”的视觉差异。

In [ ]:
source_planes = bit_plane_slicing(source_image)

# 显示 8 个位平面（按 bit7 到 bit0）
fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for index, bit in enumerate(range(7, -1, -1)):
    row = index // 4
    col = index % 4
    axes[row, col].imshow(source_planes[bit], vmin=0, vmax=1)
    axes[row, col].set_title(f'bit{bit} 平面')
    axes[row, col].axis('off')

plt.suptitle('源图 8 个比特平面（高位到低位）', y=1.02)
plt.tight_layout()
plt.show()

# 高位重建（bit7~bit4）和低位重建（bit3~bit0）
high_bits_recon = reconstruct_from_bits(source_planes, [4, 5, 6, 7])
low_bits_recon = reconstruct_from_bits(source_planes, [0, 1, 2, 3])

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
axes[0].imshow(source_image, vmin=0, vmax=255)
axes[0].set_title('原图')
axes[0].axis('off')

axes[1].imshow(high_bits_recon, vmin=0, vmax=255)
axes[1].set_title('仅高位重建（bit7~bit4）')
axes[1].axis('off')

axes[2].imshow(low_bits_recon, vmin=0, vmax=255)
axes[2].set_title('仅低位重建（bit3~bit0）')
axes[2].axis('off')

plt.tight_layout()
plt.show()

# 打印每个位平面的统计信息
print(f'{"位平面":>8} {"1比例":>10} {"二值熵":>10}')
for bit in range(7, -1, -1):
    ratio_one = np.mean(source_planes[bit])
    entropy_value = binary_entropy(source_planes[bit])
    print(f'bit{bit:>1} {ratio_one:>12.4f} {entropy_value:>10.4f}')

### 3.6 实验2：手动实现直方图规定化（说明）
本单元不调用现成“匹配接口”，手写以下流程：
1. 手动统计直方图；
2. 手动计算 PDF 与 CDF；
3. 根据 CDF 最近邻原则构造灰度映射表；
4. 应用映射得到匹配图像。

In [ ]:
def manual_histogram(image_uint8, levels=256):
    """手动统计灰度直方图（不使用 np.histogram）。"""
    histogram = np.zeros(levels, dtype=np.int64)
    flat = image_uint8.ravel()
    for value in flat:
        histogram[int(value)] += 1
    return histogram


def pdf_from_histogram(histogram):
    """由直方图得到 PDF。"""
    total = np.sum(histogram)
    if total == 0:
        return np.zeros_like(histogram, dtype=np.float64)
    return histogram.astype(np.float64) / float(total)


def cdf_from_pdf(pdf):
    """手动累加计算 CDF（不直接调用 np.cumsum）。"""
    cdf = np.zeros_like(pdf, dtype=np.float64)
    cumulative = 0.0
    for i, value in enumerate(pdf):
        cumulative += float(value)
        cdf[i] = cumulative
    return cdf


def build_mapping_by_cdf(source_cdf, target_cdf):
    """根据 CDF 最近邻原则构建 0~255 的映射表。"""
    mapping = np.zeros(256, dtype=np.uint8)

    for src_gray in range(256):
        distance = np.abs(target_cdf - source_cdf[src_gray])
        mapping[src_gray] = np.argmin(distance)

    return mapping


def apply_mapping(image_uint8, mapping):
    """按映射表逐像素完成灰度变换。"""
    flat = image_uint8.ravel()
    out = np.zeros_like(flat, dtype=np.uint8)
    for index, value in enumerate(flat):
        out[index] = mapping[int(value)]
    return out.reshape(image_uint8.shape)

### 3.7 实验2执行：完成影调迁移与可视化对比（说明）
本单元输出：
- 映射前后图像对比；
- 源图、目标图、匹配后图像的直方图与 CDF 对比；
- 简单统计指标，辅助判断匹配效果。

In [ ]:
# 计算源图与目标图的直方图、PDF、CDF
hist_source = manual_histogram(source_image)
hist_target = manual_histogram(target_image)

pdf_source = pdf_from_histogram(hist_source)
pdf_target = pdf_from_histogram(hist_target)

cdf_source = cdf_from_pdf(pdf_source)
cdf_target = cdf_from_pdf(pdf_target)

# 构建灰度映射并应用到源图
mapping_table = build_mapping_by_cdf(cdf_source, cdf_target)
matched_image = apply_mapping(source_image, mapping_table)

# 计算匹配后图像的分布
hist_matched = manual_histogram(matched_image)
pdf_matched = pdf_from_histogram(hist_matched)
cdf_matched = cdf_from_pdf(pdf_matched)

# 图像视觉对比
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(source_image, vmin=0, vmax=255)
axes[0].set_title('源图')
axes[0].axis('off')

axes[1].imshow(target_image, vmin=0, vmax=255)
axes[1].set_title('目标图')
axes[1].axis('off')

axes[2].imshow(matched_image, vmin=0, vmax=255)
axes[2].set_title('匹配后图像（影调迁移结果）')
axes[2].axis('off')

plt.tight_layout()
plt.show()

# 直方图对比
x = np.arange(256)
plt.figure(figsize=(12, 4))
plt.plot(x, hist_source, label='源图直方图', linewidth=1.2)
plt.plot(x, hist_target, label='目标图直方图', linewidth=1.2)
plt.plot(x, hist_matched, label='匹配后直方图', linewidth=1.2)
plt.title('直方图对比：源图 / 目标图 / 匹配后')
plt.xlabel('灰度级')
plt.ylabel('像素数量')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

# CDF 对比
plt.figure(figsize=(12, 4))
plt.plot(x, cdf_source, label='源图 CDF', linewidth=2)
plt.plot(x, cdf_target, label='目标图 CDF', linewidth=2)
plt.plot(x, cdf_matched, label='匹配后 CDF', linewidth=2)
plt.title('CDF 对比：匹配后是否逼近目标分布')
plt.xlabel('灰度级')
plt.ylabel('累计概率')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

# 给出简单统计指标
mean_source = np.mean(source_image)
mean_target = np.mean(target_image)
mean_matched = np.mean(matched_image)
std_source = np.std(source_image)
std_target = np.std(target_image)
std_matched = np.std(matched_image)

print('均值对比（越接近目标越好）:')
print(f'源图={mean_source:.2f}, 目标图={mean_target:.2f}, 匹配后={mean_matched:.2f}')
print('标准差对比（反映整体对比度范围）:')
print(f'源图={std_source:.2f}, 目标图={std_target:.2f}, 匹配后={std_matched:.2f}')

print('映射表示例（前 20 个灰度级）:')
for i in range(20):
    print(f'{i:3d} -> {mapping_table[i]:3d}')

## 四、总结与思考

### 实验结论
1. 比特平面分层表明：高位平面决定图像主体结构，低位平面更多包含细节扰动与微小变化。
2. 通过手写直方图、PDF、CDF 和映射表，可以完整复现直方图规定化流程。
3. 直方图规定化能够实现“影调迁移”：匹配后图像的 CDF 会显著逼近目标图像 CDF。
4. 规定化并不保证语义内容变化，只改变灰度分布特征与视觉风格。

### 思考题
1. 为什么有时 CDF 已较接近目标，但视觉上仍觉得“不像目标图”？
2. 若两幅图语义差异很大（如人脸与建筑），规定化可能产生哪些副作用？
3. 如果将映射表改为分段线性函数，结果会更平滑还是更失真？
4. 彩色图像应如何扩展：对 RGB 分通道匹配，还是先转到亮度通道匹配？